## Features and rewards of your own

An environment's market block can be the candles or what a function of the bars computes,
and its reward any function of the accounts' state. Here three features, a reward that
charges drawdown and turnover, and discrete actions, short, flat or long; then what the
agent spec records of each, and a backtest that refuses a feature function changed since
training. Bitcoin's hourly perpetual at Bybit, trained on January 2024 to June 2025.

In [1]:
import pathlib

import exchange_router
import numpy
import probatum
import probatum.sb3
import stable_baselines3
import stable_baselines3.common.vec_env
import torch

print(f"probatum {probatum.__version__}, exchange-router {exchange_router.__version__}, "
      f"stable-baselines3 {stable_baselines3.__version__}")

probatum 4.0.0, exchange-router 4.0.0, stable-baselines3 2.9.2


### The bars

In [2]:
PATH = pathlib.Path("../bars/bybit-linear-btcusdt-ethusdt-1h-2024-2025")

if PATH.exists():
    perp = probatum.Data.load(path=PATH)
else:
    with exchange_router.Router.local(exchanges=["bybit"]) as router:
        perp = probatum.Data.from_router(router, exchange="bybit", market="linear",
                                         interval="1h", symbols=["BTCUSDT", "ETHUSDT"],
                                         start="2024-01-01", end="2025-12-31", funding=True)
    perp.save(path=PATH)

train, test = perp["BTCUSDT"].split(when="2025-07-01")

### The features and the reward

The features are a function of a Data returning `(T, N, F)`, computed once over its rows.
The reward is called once a step for the whole batch with the accounts' state, every field
an array: here the log return, less the square of the drawdown and a charge on turnover.

In [3]:
def features(data):
    close   = data.closes
    returns = numpy.log(close / probatum.ta.shift(close))
    return numpy.stack([probatum.ta.roc(close, length=24),
                        probatum.ta.rsi(close, length=14),
                        probatum.ta.stdev(returns, length=24)], axis=-1)


def reward(state):
    return state.log_return - 0.5 * state.drawdown ** 2 - 0.0002 * state.turnover


env = probatum.env(train, venue="bybit_perp", window=24, features=features,
                   normalise=True, actions="discrete", reward=reward, num_envs=16,
                   episode_length=24 * 14, seed=0)

print(env.single_observation_space)
print(env.single_action_space)

Box(-inf, inf, (80,), float32)
Discrete(3)


### What the spec records

A function is never pickled. The spec records a feature function by its name, its width,
a hash of its source and a hash of its output on probe bars built the same way on every
machine, and the features' mean and deviation over the training span, which `normalise`
writes each observation over.

In [4]:
spec = env.agent_spec

{key: spec[key] for key in ["features", "normalisation", "actions", "decode", "reward"]}

{'features': {'custom': 'features',
  'width': 3,
  'source_hash': 'b2:464cab15bcc9cd48',
  'reference': 'b2:078eacc352540fe2'},
 'normalisation': {'mean': [0.19764170935776168,
   51.19739533798919,
   0.0048453654474678665],
  'deviation': [2.601440085352461, 11.81525433265113, 0.0024443507976312086]},
 'actions': {'space': 'discrete', 'choices': [3], 'max_leverage': 1.0},
 'decode': {'builtin': 'discrete'},
 'reward': {'custom': 'reward', 'source_hash': 'b2:be35fc2825e4f440'}}

### A short training

PPO with discrete actions, 150,000 steps; the observation is normalised by the environment
itself, so only SB3's monitor, which logs the episodes, sits between them.

In [5]:
torch.manual_seed(0)

agent = stable_baselines3.PPO(policy="MlpPolicy",
                              env=stable_baselines3.common.vec_env.VecMonitor(
                                  probatum.sb3.wrap(env)),
                              n_steps=512, batch_size=256, seed=0, device="cpu", verbose=0)
agent.learn(total_timesteps=150_000)

episodes = [episode["r"] for episode in agent.ep_info_buffer]
print(f"{agent.num_timesteps:,} steps, the last {len(episodes)} episodes' mean reward "
      f"{numpy.mean(episodes):.4f}")

155,648 steps, the last 100 episodes' mean reward -1.4260


### The backtest

The feature function is passed again, and held to the output its spec recorded before any
bar runs. Its weights at every close of the test bars, counted, then the run they made.

In [6]:
result = probatum.backtest(agent, data=test, venue="bybit_perp", features=features)

print(numpy.unique(result.policy[:, 0][~numpy.isnan(result.policy[:, 0])],
                   return_counts=True))

result

(array([-1.,  0.,  1.]), array([1068, 2170, 1177]))


,run,buy_and_hold,low,high,level,method,why_none
total_return_pct,-46.58,-21.45,,,,,1
cagr_pct,,,,,,,2
volatility_pct,29.57,40.07,,,,,1
sharpe,-4.06,-0.99,-6.79,-1.33,0.95,"delta method, Newey-West lag 14, floored at lag zero",
max_drawdown_pct,48.94,35.89,,,,,1
exposure_pct,50.85,,,,,,
num_trades,244,,,,,,
win_rate,0.38,,0.32,0.44,0.95,percentile bootstrap of round trips,
profit_factor,0.54,,0.37,0.80,0.95,BCa bootstrap of round trips,
costs,"2,380",,,,,,


### A changed function, refused

The same name with one length changed gives another output on the probe bars, and the
backtest refuses it with both hashes.

In [7]:
def features(data):
    close   = data.closes
    returns = numpy.log(close / probatum.ta.shift(close))
    return numpy.stack([probatum.ta.roc(close, length=48),
                        probatum.ta.rsi(close, length=14),
                        probatum.ta.stdev(returns, length=24)], axis=-1)


try:
    probatum.backtest(agent, data=test, venue="bybit_perp", features=features)
except probatum.SpecError as error:
    print(f"{type(error).__name__}: {error}")

SpecError: features: features gives b2:7144719317f67c15 on its probe, and the agent trained on features, which gave b2:078eacc352540fe2; pass the function it trained with
